# Лабораторная работа №13: Fine-tuning diffusion model

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Подготовить изображения/captions и fixed-seed baseline.
- Выполнить SD1.5 LoRA fine-tuning.
- Сравнить checkpoints, diversity и memorization.

## Теоретическая часть

Latent diffusion добавляет шум в latent space VAE, а U-Net предсказывает noise/velocity. LoRA адаптирует attention projections без изменения всех весов. Фиксированные seeds и prompts обязательны для сравнения checkpoints.

![Архитектура latent diffusion](https://raw.githubusercontent.com/CompVis/latent-diffusion/main/assets/modelfigure.png)


Latent diffusion обучает UNet предсказывать ε в пространстве VAE. LoRA добавляет обучаемые низкоранговые поправки к attention projections; VAE, text encoder и base UNet остаются замороженными. CLIP text similarity, разнообразие и ближайшая train генерация измеряют разные свойства и требуют ручного дополнения.

Для LoRA $W^{\prime}=W+BA$, где $B\in\mathbb{R}^{d\times r}$, $A\in\mathbb{R}^{r\times k}$, $r\ll\min(d,k)$. Noise-prediction минимизирует $\mathbb{E}_{x_0,\epsilon,t}\|\epsilon-\epsilon_\theta(z_t,t,c)\|_2^2$ на VAE-latents.

### Экспериментальный протокол

Основной эксперимент: SD 1.5 и 28 открытых реальных фотографий бабочек Wikimedia Commons при 512 px. `download_open_data.py` скачивает файлы и для каждого проверяет метаданные CC0-1.0 через Commons API, фиксирует страницу источника, URL лицензии и SHA256. Никакие файлы студента не требуются; `CV_LORA_DATA` задаёт альтернативный каталог с той же структурой metadata.jsonl. 3 независимых адаптера по 400 optimizer updates, checkpoint на 200/400; фиксированные held-out prompts и seeds. FAST_DEV_RUN: 256 px, один update, только smoke. Требуются GPU, Pillow, pip install diffusers transformers accelerate peft safetensors torchvision; на CPU не подменять обучение имитацией. CV_ARTIFACTS указывает на постоянный каталог Drive.

### Материалы

- [Diffusers LoRA training](https://huggingface.co/docs/diffusers/training/lora)
- [Latent Diffusion](https://arxiv.org/abs/2112.10752)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.

Версии библиотек, источник данных и лицензия должны быть указаны в отчёте.

## Реализация

`FAST_DEV_RUN=True` проверяет связность пайплайна на малом бюджете и не является итоговым экспериментом. Полная сдача требует `FAST_DEV_RUN=False` и реально полученных метрик.

In [ ]:
from pathlib import Path
import os, json, random, time
import numpy as np
import torch
SEED = 42
FAST_DEV_RUN = False  # True только для короткой проверки связности, не для сдачи.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACTS = Path(os.environ.get('CV_ARTIFACTS', 'artifacts'))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def save_json(name, value):
    destination = ARTIFACTS / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, ensure_ascii=False, indent=2))
print({'device':DEVICE,'fast_dev_run':FAST_DEV_RUN,'artifacts':str(ARTIFACTS)})

### Лицензия и валидация изображений

*Вес: 1 балл.*

**Постановка.** При малом наборе изображений утечка оценки в train captions делает сравнение LoRA некорректным. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Проверьте физические файлы, RGB-декодирование, уникальность и лицензии подписей.

**Результат.** Проверяемое состояние «Лицензия и валидация изображений» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Проверьте физические файлы, RGB-декодирование, уникальность и лицензии подписей» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Отложенные текстовые запросы

*Вес: 1 балл.*

**Постановка.** При малом наборе изображений утечка оценки в train captions делает сравнение LoRA некорректным. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Зафиксируйте prompts и seeds, исключите точное совпадение с обучающими подписями.

**Результат.** Проверяемое состояние «Отложенные текстовые запросы» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Зафиксируйте prompts и seeds, исключите точное совпадение с обучающими подписями» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Baseline SD на фиксированных запросах

*Вес: 1 балл.*

**Постановка.** Baseline до обучения и сопоставимый inference protocol позволяют увидеть фактический вклад LoRA. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите исходную SD1.5 на GPU и сгенерируйте baseline до установки адаптера на фиксированных запросах/seeds.

**Результат.** Проверяемое состояние «Baseline SD на фиксированных запросах» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите исходную SD1.5 на GPU и сгенерируйте baseline до установки адаптера на фиксированных запросах/seeds» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Подготовка пар изображение–caption

*Вес: 1 балл.*

**Постановка.** Различия rank и learning rate оцениваются при одинаковых изображениях, количестве optimizer updates и eval prompts. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Соберите Dataset с препроцессингом и токенизацией; проверьте длину и диапазон.

**Результат.** Проверяемое состояние «Подготовка пар изображение–caption» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Соберите Dataset с препроцессингом и токенизацией; проверьте длину и диапазон» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Заморозка основы и конфигурации LoRA

*Вес: 1 балл.*

**Постановка.** Различия rank и learning rate оцениваются при одинаковых изображениях, количестве optimizer updates и eval prompts. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Заморозьте основу, установите тренировочный schedule, выберите три конфигурации rank/LR.

**Результат.** Проверяемое состояние «Заморозка основы и конфигурации LoRA» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Заморозьте основу, установите тренировочный schedule, выберите три конфигурации rank/LR» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Оптимизация адаптеров и checkpoint

*Вес: 1 балл.*

**Постановка.** Различия rank и learning rate оцениваются при одинаковых изображениях, количестве optimizer updates и eval prompts. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Обучите независимые адаптеры, сохраняйте optimizer, scaler и переносимые веса при каждом контрольном шаге.

**Результат.** Проверяемое состояние «Оптимизация адаптеров и checkpoint» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Обучите независимые адаптеры, сохраняйте optimizer, scaler и переносимые веса при каждом контрольном шаге» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Признаки CLIP для независимой оценки

*Вес: 1 балл.*

**Постановка.** Одна similarity-метрика не определяет качество генерации: высокое совпадение с текстом может сопровождаться копированием train. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Загрузите CLIP и определите нормализованные текстовые и визуальные embeddings.

**Результат.** Проверяемое состояние «Признаки CLIP для независимой оценки» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Загрузите CLIP и определите нормализованные текстовые и визуальные embeddings» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Поиск ближайших обучающих изображений

*Вес: 1 балл.*

**Постановка.** Одна similarity-метрика не определяет качество генерации: высокое совпадение с текстом может сопровождаться копированием train. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Вычислите эмбеддинги реальных train изображений и зафиксируйте набор checkpoint для оценки.

**Результат.** Проверяемое состояние «Поиск ближайших обучающих изображений» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Вычислите эмбеддинги реальных train изображений и зафиксируйте набор checkpoint для оценки» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Парные генерации и измеримые показатели

*Вес: 1 балл.*

**Постановка.** Одна similarity-метрика не определяет качество генерации: высокое совпадение с текстом может сопровождаться копированием train. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Для равных prompts/seeds измерьте текстовое сходство, разнообразие и ближайший train объект; baseline не должен содержать LoRA.

**Результат.** Проверяемое состояние «Парные генерации и измеримые показатели» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Для равных prompts/seeds измерьте текстовое сходство, разнообразие и ближайший train объект; baseline не должен содержать LoRA» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Слепая таблица оценки

*Вес: 1 балл.*

**Постановка.** Одна similarity-метрика не определяет качество генерации: высокое совпадение с текстом может сопровождаться копированием train. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Подготовьте перемешанный список изображений с пустыми оценками для реальных экспертов.

**Результат.** Проверяемое состояние «Слепая таблица оценки» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Подготовьте перемешанный список изображений с пустыми оценками для реальных экспертов» и его результат доступен следующему этапу.

In [ ]:
# Реализуйте это задание по постановке выше.
# Сохраните перечисленные артефакты и выведите фактически измеренные значения.
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Творческое мини-исследование

*Вес: 1 балл, бонус сверх десяти обязательных заданий.*

**Постановка.** Контролируемая абляция позволяет отделить влияние одного фактора от шума выборки. Гипотеза формулируется до просмотра отложенных результатов.

Выполните следующие действия:

1. Сформулируйте собственную предметную гипотезу о поведении модели и измените один фактор, сохраняя те же реальные входы, seed и вычислительный бюджет.
2. Сравните с основным экспериментом, не используя test для выбора настройки; объясните наблюдаемые расхождения.

**Результат.** Отдельные код, измеренная таблица и интерпретация в отчёте.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена на реальных входах, сохранены фактические измерения и показана контролируемая разница. Бонус не является обязательным для Run All.

## Анализ результатов

Сопоставьте конфигурации при одинаковых данных, seeds и бюджете. Приведите таблицу фактических метрик и времени, не менее пяти неудачных примеров с объяснением причин, ограничения и проверяемые выводы. Не подменяйте измерения утверждением об успешном запуске.

## Требования к сдаче

Перед сдачей перезапустите ядро и выполните все ячейки по порядку в полном режиме. Приложите конфигурацию, версии библиотек, источники/лицензии данных и модели, checkpoints, таблицу измерений, изображения и краткий анализ ошибок. Для длительного обучения перенесите каталог `artifacts/` на Google Drive и возобновляйте обучение с последнего checkpoint; отсутствие GPU или данных нельзя заменять синтетическими метриками.